## 5, Calculations to compute:
- Base_Fare
- Per_Km_Rate
- Per_Minute_Rate
- Trip_Duration_Minutes

### UI only takes Passenger_Count input so basing calculations on that

In [1]:
import pandas as pd
import seaborn as sns

In [3]:
df_calc = pd.read_csv("df_BI_copy.csv")
df_calc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 875 entries, 0 to 874
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Trip_Distance_km        875 non-null    float64
 1   Passenger_Count         875 non-null    float64
 2   Base_Fare               875 non-null    float64
 3   Per_Km_Rate             875 non-null    float64
 4   Per_Minute_Rate         875 non-null    float64
 5   Trip_Duration_Minutes   875 non-null    float64
 6   Trip_Price              875 non-null    float64
 7   IsBusinessHour          875 non-null    int64  
 8   IsRain                  875 non-null    int64  
 9   IsSnow                  875 non-null    int64  
 10  IsWeekend               875 non-null    int64  
 11  Traffic_Conditions_Num  875 non-null    int64  
 12  Time_of_Day_Num         875 non-null    int64  
dtypes: float64(7), int64(6)
memory usage: 89.0 KB


In [4]:
df_calc["Base_Fare"].median()

np.float64(3.508800959232614)

In [5]:
df_calc["Per_Km_Rate"].median()

np.float64(1.2363592814371256)

In [6]:
df_calc["Per_Minute_Rate"].median()

np.float64(0.292647412755716)

In [7]:
# calculate median by passenger
med_by_pax = (
    df_calc.groupby("Passenger_Count")[["Base_Fare","Per_Km_Rate","Per_Minute_Rate","Trip_Duration_Minutes"]].median()
)
med_by_pax

,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes
Passenger_Count,,,,
1.0,3.508801,1.225000,0.292647,61.96
2.0,3.315000,1.190000,0.290000,61.96
3.0,3.540000,1.236359,0.292647,61.96
4.0,3.590000,1.300000,0.292647,62.05


In [8]:
mins_per_km = (df_calc["Trip_Duration_Minutes"] / df_calc["Trip_Distance_km"]).median()
mins_per_km

np.float64(2.4403769481696265)

In [9]:
df_calc.groupby(["Passenger_Count", "IsWeekend"])["Trip_Price"].mean()

Passenger_Count  IsWeekend
1.0              0            53.207187
                 1            50.308430
2.0              0            50.920594
                 1            45.366025
3.0              0            51.369204
                 1            53.621745
4.0              0            56.435668
                 1            54.350605
Name: Trip_Price, dtype: float64

In [10]:
col = "IsRain"
g = df_calc.groupby(col)["Trip_Price"].mean()
m0, m1 = g.get(0, float("nan")), g.get(1, float("nan"))
uplift_percent = ((m1 - m0) / m0) * 100
print(f"{col}: mean(0)={m0:.2f}, mean(1)={m1:.2f}, uplift={uplift_percent:.2f}%")


IsRain: mean(0)=51.97, mean(1)=53.14, uplift=2.24%


In [12]:
binary_cols = ["IsBusinessHour", "IsRain", "IsSnow", "IsWeekend"]

rows = []
for col in binary_cols:
    g = df_calc.groupby(col)["Trip_Price"].mean()
    m0, m1 = g.get(0, float("nan")), g.get(1, float("nan"))
    uplift = ((m1 - m0) / m0) * 100
    rows.append({"feature": col, "mean_0": m0, "mean_1": m1, "uplift_percent": uplift})

uplift_df = pd.DataFrame(rows)
uplift_df

,feature,mean_0,mean_1,uplift_percent
0,IsBusinessHour,52.186728,52.550568,0.697188
1,IsRain,51.972883,53.138489,2.242719
2,IsSnow,52.192989,53.913840,3.297094
3,IsWeekend,52.797765,51.157111,-3.107431


In [15]:
summary = (
    df_calc.groupby("IsRain")
      .agg(mean_price=("Trip_Price", "mean"),
           count=("Trip_Price", "size"))
)
summary

,mean_price,count
IsRain,,
0,51.972883,633
1,53.138489,242


In [17]:
seg = (
    df_calc.groupby(["Passenger_Count", "IsWeekend"])["Trip_Price"]
      .mean()
      .unstack("IsWeekend")  # columns 0/1
      .rename(columns={0: "mean_0", 1: "mean_1"})
)
seg["uplift_percent"] = ((seg["mean_1"] - seg["mean_0"]) / seg["mean_0"]) * 100
seg

IsWeekend,mean_0,mean_1,uplift_percent
Passenger_Count,,,
1.0,53.207187,50.308430,-5.448057
2.0,50.920594,45.366025,-10.908296
3.0,51.369204,53.621745,4.385003
4.0,56.435668,54.350605,-3.694585


In [18]:
def uplift_for_flag(data, flag_col, target="Trip_Price"):
    g = data.groupby(flag_col)[target].mean()
    m0, m1 = g.get(0, float("nan")), g.get(1, float("nan"))
    return pd.Series({
        "mean_0": m0,
        "mean_1": m1,
        "uplift_percent": ((m1 - m0) / m0) * 100
    })

uplift_for_flag(df_calc, "IsBusinessHour")

mean_0            52.186728
mean_1            52.550568
uplift_percent     0.697188
dtype: float64